In [11]:
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Optional
from google.colab import files


In [ ]:
# Helper

def export_df_to_excel(
    df: pd.DataFrame,
    file_path: str,
    sheet_name: str = "data",
    index: bool = True,
):
    df.to_excel(
        file_path,
        sheet_name=sheet_name,
        index=index
    )


# 1 - Importer Indice historique


In [9]:
def load_excel_file():
    """
    Importe un fichier Excel depuis Colab et retourne :
    - le DataFrame complet
    - la colonne Date convertie en datetime
    - la colonne Index convertie en numérique
    """

    uploaded = files.upload()
    filename = next(iter(uploaded))

    df = pd.read_excel(filename)

    dates = pd.to_datetime(df["Date"])

    index = pd.to_numeric(
        df["Index"].astype(str).str.replace(",", "."),
        errors="coerce"
    )

    return df, dates, index

In [10]:
df1, dates1, index1 = load_excel_file()
# df2, dates2, index2 = load_excel_file()
# df3, dates3, index3 = load_excel_file()

print(df1.head())
#print(df2.head())

Saving Data_SAF.xlsx to Data_SAF (3).xlsx
        Date        Index
0 2005-12-29  1000.000000
1 2005-12-30   991.560128
2 2006-01-03   997.860255
3 2006-01-04   999.580579
4 2006-01-05  1000.846722


# 2 - Traitement de Time Serie


In [13]:
def load_index_series_df(df: pd.DataFrame) -> pd.Series:
    """ load_index_series_AB()
    Transforme un DataFrame contenant :
      - Colonne 0 : Date
      - Colonne 1 : Index level

    Retourne :
      pd.Series avec DatetimeIndex et valeurs float.
    """

    date_col = df.columns[0]
    level_col = df.columns[1]

    # Parse date
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")

    # Parse level (format français)
    lvl = (
        df[level_col]
        .astype(str)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False)
    )

    df[level_col] = pd.to_numeric(lvl, errors="coerce")

    s = (
        df[[date_col, level_col]]
        .dropna()
        .drop_duplicates(subset=date_col)
        .set_index(date_col)[level_col]
        .sort_index()
    )

    if s.empty:
        raise ValueError("Series vide après nettoyage.")

    if (s <= 0).any():
        raise ValueError("Certaines valeurs sont <= 0.")

    return s

In [16]:
levels = load_index_series_df(df1)
print(levels)

Date
2005-12-29    1000.000000
2005-12-30     991.560128
2006-01-03     997.860255
2006-01-04     999.580579
2006-01-05    1000.846722
                 ...     
2025-05-08    2621.333962
2025-05-09    2633.512877
2025-05-12    2715.116609
2025-05-13    2740.053076
2025-05-14    2748.187278
Name: Index, Length: 4974, dtype: float64


# 3 - Test BS


In [21]:
r_neutre_cc = 0.0691009521484375

params = build_bs_params_simple(
    levels=levels,
    start_date="2025-12-31",
    r_neutre_annual_cc=r_neutre_cc,
    s0_default=1000,
    use_real_s0=False,
)

print(params)

BSParams(r_annual_cc=0.0691009521484375, sigma_annual=0.2558122680177892, s0=1000.0, s0_date=None)


# Black&Schole Model


### main

In [19]:

@dataclass
class BSParams:
    r_annual_cc: float
    sigma_annual: float
    s0: float
    s0_date: Optional[pd.Timestamp]


def estimate_sigma_from_history(levels: pd.Series, day_count: int = 365) -> float:
    """sigma_annual = std(log-return daily) * sqrt(day_count)"""
    s = levels.sort_index()
    logrets = np.log(s / s.shift(1)).dropna()
    return float(logrets.std(ddof=1) * np.sqrt(day_count))


def get_s0(
    levels: pd.Series,
    start_date: str,
    default_s0: float = 1000.0,
    use_real_if_available: bool = False,
) -> tuple[float, Optional[pd.Timestamp]]:
    """Chọn S0: default hoặc lấy đúng level tại start_date nếu có."""
    if not use_real_if_available:
        return float(default_s0), None

    d = pd.Timestamp(start_date)
    if d in levels.index:
        return float(levels.loc[d]), d
    return float(default_s0), None


def build_bs_params_simple(
    levels: pd.Series,
    start_date: str,
    r_neutre_annual_cc: float,
    day_count: int = 365,
    s0_default: float = 1000.0,
    use_real_s0: bool = False,
) -> BSParams:
    """
    Build params đơn giản:
      - sigma: từ dữ liệu quá khứ
      - r_cc: = r_neutre (bạn truyền vào)
      - s0: default hoặc lấy level thật tại start_date nếu có
    """
    s0, s0_date = get_s0(
        levels,
        start_date=start_date,
        default_s0=s0_default,
        use_real_if_available=use_real_s0,
    )

    sigma = estimate_sigma_from_history(levels, day_count=day_count)

    return BSParams(
        r_annual_cc=float(r_neutre_annual_cc),
        sigma_annual=float(sigma),
        s0=float(s0),
        s0_date=s0_date,
    )


def simulate_gbm_monthly(
    s0: float,
    r_annual_cc: float,
    sigma_annual: float,
    start_date: str,
    n_months: int,
    n_sims: int,
    seed: Optional[int] = 42,
) -> pd.DataFrame:
    """GBM monthly simulation, output scenario x dates."""
    rng = np.random.default_rng(seed)

    dt = 1.0 / 12.0
    dates = pd.date_range(start=pd.Timestamp(start_date), periods=n_months + 1, freq="M")

    Z = rng.standard_normal(size=(n_months, n_sims))
    drift = (r_annual_cc - 0.5 * sigma_annual**2) * dt
    diffusion = sigma_annual * np.sqrt(dt) * Z

    log_paths = np.vstack([np.zeros((1, n_sims)), np.cumsum(drift + diffusion, axis=0)])
    paths = s0 * np.exp(log_paths)

    df = pd.DataFrame(paths.T, index=np.arange(1, n_sims + 1), columns=dates)
    df.index.name = "scenario"
    return df
